# 01 — Shared State Bugs: Race Conditions Live

**Goal:** See race conditions happen. Understand WHY threads are dangerous with shared state.

## Exercise 1.1 — Break a counter

Create a global `counter = 0`. Write `increment(n)` that does `counter += 1` in a loop `n` times.

Run 10 threads, each calling `increment(100_000)`. Expected total: 1,000,000.

Print the actual result. It will be LESS than 1,000,000.

**Why:** `counter += 1` is NOT atomic. It's actually:
```
temp = counter    ← read
temp = temp + 1   ← modify  
counter = temp    ← write
```
The OS can switch threads between ANY of these steps. Two threads read the same value, both write back +1, and one increment is lost.

**This CANNOT happen with asyncio** — between two `await`s, you run alone.

In [ ]:
import threading
import time

# Exercise 1.1 — your code here


## Exercise 1.2 — Fix with Lock

Create a `threading.Lock()`. Wrap the `counter += 1` line with `with lock:`.

Run the same 10-thread test. Verify the count is exactly 1,000,000.

**Think about:** The lock serializes ALL increments — only one thread runs the critical section at a time. Is threaded + lock faster or slower than single-threaded?

In [ ]:
# Exercise 1.2 — your code here


## Exercise 1.3 — Deadlock

Create two locks: `lock_a` and `lock_b`.

Write `worker_1()`: acquires lock_a, sleeps 0.1s, then tries to acquire lock_b.  
Write `worker_2()`: acquires lock_b, sleeps 0.1s, then tries to acquire lock_a.

**Use `lock.acquire(timeout=2)` instead of `with lock:` so you don't actually hang forever.**

If `acquire` returns `False`, print "DEADLOCK detected".

**Prevention:** Always acquire locks in the same order. If everyone acquires lock_a before lock_b, no deadlock.

In [ ]:
# Exercise 1.3 — your code here


## Exercise 1.4 — Find and fix the race condition

This bank transfer has a race condition:
```python
accounts = {"Alice": 1000, "Bob": 1000}

def transfer(from_acct, to_acct, amount):
    if accounts[from_acct] >= amount:     # check
        time.sleep(0.001)                  # OS can switch HERE
        accounts[from_acct] -= amount      # act
        accounts[to_acct] += amount
```

Run 100 threads doing `transfer("Alice", "Bob", 10)` and 100 doing `transfer("Bob", "Alice", 10)` concurrently.

Check: `Alice + Bob` should always equal 2000. Does it?

Fix it with a Lock so the total is always conserved.

In [ ]:
# Exercise 1.4 — your code here


## Summary

| Bug | Cause | Fix |
|-----|-------|-----|
| **Race condition** | Two threads read/write same variable | Lock |
| **Deadlock** | Two threads each hold a lock the other needs | Consistent lock ordering |
| **Lost updates** | Read-modify-write interrupted | Atomic operations or Lock |

**Why asyncio is easier:** Between two `await`s, you're alone. No races possible within a single coroutine.

**Next notebook:** Thread safety primitives (RLock, Event, Semaphore)